In [1]:
import nba_api.stats.endpoints
import requests
import json
import pandas as pd
import nba_api

In [2]:
# Get Timberwolves player game logs for the season
wolves_player_logs = nba_api.stats.endpoints.PlayerGameLogs(season_nullable='2025-26',team_id_nullable='1610612750').get_data_frames()[0]

wolves_player_logs

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT
0,2025-26,1630162,Anthony Edwards,Anthony,1610612750,MIN,Minnesota Timberwolves,0022500727,2026-02-04T00:00:00,MIN @ TOR,...,23,29,412,7,47,3,17,1,37:46,1
1,2025-26,1630538,Bones Hyland,Bones,1610612750,MIN,Minnesota Timberwolves,0022500727,2026-02-04T00:00:00,MIN @ TOR,...,192,95,273,93,47,3,82,1,26:02,1
2,2025-26,1629675,Naz Reid,Naz,1610612750,MIN,Minnesota Timberwolves,0022500727,2026-02-04T00:00:00,MIN @ TOR,...,192,140,283,147,47,3,127,1,28:39,1
3,2025-26,1628978,Donte DiVincenzo,Donte,1610612750,MIN,Minnesota Timberwolves,0022500727,2026-02-04T00:00:00,MIN @ TOR,...,449,167,316,219,47,3,160,1,31:00,1
4,2025-26,1630183,Jaden McDaniels,Jaden,1610612750,MIN,Minnesota Timberwolves,0022500727,2026-02-04T00:00:00,MIN @ TOR,...,300,106,385,188,47,3,194,1,39:43,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653,2025-26,204060,Joe Ingles,Joe,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,449,469,283,493,47,3,474,1,16:10,1
654,2025-26,1642389,Zyon Pullin,Zyon,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,449,506,412,570,47,3,552,1,10:19,1
655,2025-26,1631262,Jules Bernard,Jules,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,449,565,429,584,47,3,588,1,4:10,1
656,2025-26,1641803,Tristen Newton,Tristen,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,449,506,283,600,47,3,588,1,3:08,1


In [3]:
# Filter for Rudy Gobert game logs
gobert_logs = wolves_player_logs[wolves_player_logs['PLAYER_NAME'] == 'Rudy Gobert'].copy()

print(f"Rudy Gobert games: {len(gobert_logs)}")
print(f"Total games in dataset: {len(gobert_logs)}")

# Display first few rows to verify
gobert_logs[['GAME_DATE', 'MATCHUP', 'WL', 'REB', 'PTS']].head(10)

Rudy Gobert games: 54
Total games in dataset: 54


,GAME_DATE,MATCHUP,WL,REB,PTS
5,2026-02-04T00:00:00,MIN @ TOR,W,12,10
15,2026-02-02T00:00:00,MIN @ MEM,L,10,7
22,2026-01-31T00:00:00,MIN @ MEM,W,16,9
33,2026-01-29T00:00:00,MIN vs. OKC,W,11,14
46,2026-01-28T00:00:00,MIN @ DAL,W,6,6
52,2026-01-26T00:00:00,MIN vs. GSW,W,17,15
70,2026-01-25T00:00:00,MIN vs. GSW,L,5,4
81,2026-01-22T00:00:00,MIN vs. CHI,L,11,10
90,2026-01-20T00:00:00,MIN @ UTA,L,10,11
111,2026-01-16T00:00:00,MIN @ HOU,L,13,10


In [4]:
# Analyze Timberwolves record based on Rudy Gobert's rebound thresholds
from IPython.display import display

print("Timberwolves Record Based on Rudy Gobert's Rebounds")
print("=" * 80)

# Define rebound thresholds
thresholds = [
    ("Fewer than 10 rebounds", lambda x: x < 10),
    ("10 or more rebounds", lambda x: x >= 10),
    ("11 or more rebounds", lambda x: x >= 11),
    ("12 or more rebounds", lambda x: x >= 12),
    ("13 or more rebounds", lambda x: x >= 13),
    ("14 or more rebounds", lambda x: x >= 14),
    ("15 or more rebounds", lambda x: x >= 15)
]

results = []

for threshold_name, threshold_func in thresholds:
    # Filter games based on threshold
    filtered_games = gobert_logs[gobert_logs['REB'].apply(threshold_func)].copy()
    
    if len(filtered_games) > 0:
        wins = (filtered_games['WL'] == 'W').sum()
        losses = (filtered_games['WL'] == 'L').sum()
        total = wins + losses
        
        if total > 0:
            win_pct = wins / total
            avg_reb = filtered_games['REB'].mean()
            
            results.append({
                'Threshold': threshold_name,
                'Games': total,
                'Wins': wins,
                'Losses': losses,
                'Win %': f"{win_pct:.3f} ({win_pct*100:.1f}%)",
                'Avg REB': f"{avg_reb:.2f}"
            })

# Create results dataframe
results_df = pd.DataFrame(results)
display(results_df)

Timberwolves Record Based on Rudy Gobert's Rebounds


,Threshold,Games,Wins,Losses,Win %,Avg REB
0,Fewer than 10 rebounds,19,9,10,0.474 (47.4%),6.84
1,10 or more rebounds,35,23,12,0.657 (65.7%),13.17
2,11 or more rebounds,32,22,10,0.688 (68.8%),13.47
3,12 or more rebounds,28,21,7,0.750 (75.0%),13.82
4,13 or more rebounds,17,14,3,0.824 (82.4%),15.00
5,14 or more rebounds,13,12,1,0.923 (92.3%),15.62
6,15 or more rebounds,9,8,1,0.889 (88.9%),16.33


In [5]:
# Detailed breakdown for each threshold
print("Detailed Breakdown by Rebound Threshold")
print("=" * 100)

for threshold_name, threshold_func in thresholds:
    filtered_games = gobert_logs[gobert_logs['REB'].apply(threshold_func)].copy()
    
    if len(filtered_games) > 0:
        wins = (filtered_games['WL'] == 'W').sum()
        losses = (filtered_games['WL'] == 'L').sum()
        total = wins + losses
        
        if total > 0:
            win_pct = wins / total
            avg_reb = filtered_games['REB'].mean()
            
            print(f"\n{threshold_name}:")
            print(f"  Games: {total}")
            print(f"  Record: {wins}-{losses}")
            print(f"  Win Percentage: {win_pct:.3f} ({win_pct*100:.1f}%)")
            print(f"  Average Rebounds: {avg_reb:.2f}")
            
            # Show game details
            display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'REB', 'PTS', 'MIN']
            display_cols = [col for col in display_cols if col in filtered_games.columns]
            
            print(f"\n  Game Logs:")
            display(filtered_games[display_cols].sort_values('GAME_DATE', ascending=False))

Detailed Breakdown by Rebound Threshold

Fewer than 10 rebounds:
  Games: 19
  Record: 9-10
  Win Percentage: 0.474 (47.4%)
  Average Rebounds: 6.84

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
46,2026-01-28T00:00:00,MIN @ DAL,W,6,6,20.600000
70,2026-01-25T00:00:00,MIN vs. GSW,L,5,4,24.100000
228,2025-12-27T00:00:00,MIN vs. BKN,L,8,6,31.433333
303,2025-12-08T00:00:00,MIN vs. PHX,L,8,15,20.983333
317,2025-12-06T00:00:00,MIN vs. LAC,W,7,4,32.700000
348,2025-11-30T00:00:00,MIN vs. SAS,W,8,8,24.033333
360,2025-11-29T00:00:00,MIN vs. BOS,W,8,12,30.095000
406,2025-11-17T00:00:00,MIN vs. DAL,W,9,15,29.100000
424,2025-11-15T00:00:00,MIN vs. DEN,L,6,4,23.166667
432,2025-11-14T00:00:00,MIN vs. SAC,W,8,11,34.331667



10 or more rebounds:
  Games: 35
  Record: 23-12
  Win Percentage: 0.657 (65.7%)
  Average Rebounds: 13.17

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
5,2026-02-04T00:00:00,MIN @ TOR,W,12,10,32.896667
15,2026-02-02T00:00:00,MIN @ MEM,L,10,7,25.150000
22,2026-01-31T00:00:00,MIN @ MEM,W,16,9,30.800000
33,2026-01-29T00:00:00,MIN vs. OKC,W,11,14,34.183333
52,2026-01-26T00:00:00,MIN vs. GSW,W,17,15,34.983333
81,2026-01-22T00:00:00,MIN vs. CHI,L,11,10,33.700000
90,2026-01-20T00:00:00,MIN @ UTA,L,10,11,29.273333
111,2026-01-16T00:00:00,MIN @ HOU,L,13,10,32.448333
134,2026-01-11T00:00:00,MIN vs. SAS,W,14,2,28.858333
142,2026-01-10T00:00:00,MIN @ CLE,L,12,8,31.400000



11 or more rebounds:
  Games: 32
  Record: 22-10
  Win Percentage: 0.688 (68.8%)
  Average Rebounds: 13.47

  Game Logs:


...

,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
22,2026-01-31T00:00:00,MIN @ MEM,W,16,9,30.800000
52,2026-01-26T00:00:00,MIN vs. GSW,W,17,15,34.983333
111,2026-01-16T00:00:00,MIN @ HOU,L,13,10,32.448333
134,2026-01-11T00:00:00,MIN vs. SAS,W,14,2,28.858333
155,2026-01-08T00:00:00,MIN vs. CLE,W,13,11,36.850000
161,2026-01-06T00:00:00,MIN vs. MIA,W,16,13,32.633333
175,2026-01-04T00:00:00,MIN @ WAS,W,14,18,29.316667
247,2025-12-23T00:00:00,MIN vs. NYK,W,16,11,37.750000
256,2025-12-21T00:00:00,MIN vs. MIL,W,18,11,39.291667
269,2025-12-19T00:00:00,MIN vs. OKC,W,14,9,33.118333



14 or more rebounds:
  Games: 13
  Record: 12-1
  Win Percentage: 0.923 (92.3%)
  Average Rebounds: 15.62

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
22,2026-01-31T00:00:00,MIN @ MEM,W,16,9,30.800000
52,2026-01-26T00:00:00,MIN vs. GSW,W,17,15,34.983333
134,2026-01-11T00:00:00,MIN vs. SAS,W,14,2,28.858333
161,2026-01-06T00:00:00,MIN vs. MIA,W,16,13,32.633333
175,2026-01-04T00:00:00,MIN @ WAS,W,14,18,29.316667
247,2025-12-23T00:00:00,MIN vs. NYK,W,16,11,37.750000
256,2025-12-21T00:00:00,MIN vs. MIL,W,18,11,39.291667
269,2025-12-19T00:00:00,MIN vs. OKC,W,14,9,33.118333
274,2025-12-17T00:00:00,MIN vs. MEM,L,16,16,35.466667
292,2025-12-12T00:00:00,MIN @ GSW,W,14,24,35.176667



15 or more rebounds:
  Games: 9
  Record: 8-1
  Win Percentage: 0.889 (88.9%)
  Average Rebounds: 16.33

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
22,2026-01-31T00:00:00,MIN @ MEM,W,16,9,30.800000
52,2026-01-26T00:00:00,MIN vs. GSW,W,17,15,34.983333
161,2026-01-06T00:00:00,MIN vs. MIA,W,16,13,32.633333
247,2025-12-23T00:00:00,MIN vs. NYK,W,16,11,37.750000
256,2025-12-21T00:00:00,MIN vs. MIL,W,18,11,39.291667
274,2025-12-17T00:00:00,MIN vs. MEM,L,16,16,35.466667
399,2025-11-19T00:00:00,MIN vs. WAS,W,15,9,36.083333
503,2025-11-01T00:00:00,MIN @ CHA,W,15,14,35.326667
536,2025-10-26T00:00:00,MIN vs. IND,W,18,14,33.998333


In [6]:
# Summary statistics
print("Rudy Gobert Rebounding Summary")
print("=" * 80)

if len(gobert_logs) > 0:
    print(f"\nTotal Games: {len(gobert_logs)}")
    print(f"Average Rebounds: {gobert_logs['REB'].mean():.2f}")
    print(f"Median Rebounds: {gobert_logs['REB'].median():.2f}")
    print(f"Min Rebounds: {gobert_logs['REB'].min()}")
    print(f"Max Rebounds: {gobert_logs['REB'].max()}")
    
    # Overall team record
    wins = (gobert_logs['WL'] == 'W').sum()
    losses = (gobert_logs['WL'] == 'L').sum()
    total = wins + losses
    
    if total > 0:
        win_pct = wins / total
        print(f"\nOverall Team Record: {wins}-{losses} ({win_pct:.3f} / {win_pct*100:.1f}%)")
    
    # Rebound distribution
    print(f"\nRebound Distribution:")
    rebound_counts = gobert_logs['REB'].value_counts().sort_index()
    print(rebound_counts)

Rudy Gobert Rebounding Summary

Total Games: 54
Average Rebounds: 10.94
Median Rebounds: 12.00
Min Rebounds: 3
Max Rebounds: 18

Overall Team Record: 32-22 (0.593 / 59.3%)

Rebound Distribution:
REB
3      1
4      1
5      2
6      4
7      2
8      6
9      3
10     3
11     4
12    11
13     4
14     4
15     2
16     4
17     1
18     2
Name: count, dtype: int64
